# RepairScope-X — Notebook 28: Prospective adaptive self-repair study
### ALoDLM-1.7B full-precision · calibration/held-out benchmark · long, restart-safe T4 experiment

**This notebook runs NEW real-model inference; no mock results count as research evidence.** It tests a hypothesis, **not a guaranteed improvement**. Main endpoints: independently recovered held-out failed problems, exact task success, and measured native decoder cost. An absence of significant improvement is a valid finding.

**Experimental stages**: (A) independent synthetic benchmark with held-out problems and genuine model-native failures; (B) calibrate 4 repair scopes on 24 calibration failures; (C) freeze learned adaptive policy; (D) prospectively test five prespecified policies on 36 distinct held-out failures at three seeds; (E) two-stage adaptive vs. repeated-global retry; (F) left1 decoder ablation; (G) statistical analysis, research tables/figures, and ZIP export.

**Resource plan:** ALoDLM-1.7B FP32 on T4 (~8 GiB observed previously); ~60+ native generations and up to ~1,164 repairs. Overall *intended* wall time 1–3 hours on a continuously available GPU, **not a guarantee**. Runs are bounded by a configurable 2 h 50 min session budget, resume from Google Drive, and export intermediate results. Long sweeps use single-digit trials per cell. Colab Free GPU and Drive connectivity may interrupt or prolong runs.

**No leakage:** the policy is fit exclusively on calibration failures. Evaluation reference values are never used by the adaptive policy for word problems; Countdown includes the target as part of the task and can be verified exactly. This study uses **procedurally generated math prompts**, *not* held-out GSM8K/SVAMP, and is not a generalization claim for those datasets. Existing ALoDLM results are historical context, not pooled here.

**Fairness:** primary comparisons match the number of *proposals*, not exact underlying FLOPs; decoder's `loop_token_passes`, recurrent passes, wall seconds and VRAM are reported separately. Native suffix continuation is **not arbitrary masked-token infilling**. Token preservation is assessed from the exact source tokens, not re-tokenized text. A fixed train/test split and frozen policy avoid test-label tuning. No test-time oracle chooses a policy or candidate.

## 0. Installation and source pin — run once on a fresh Google Colab T4
The Git commit is the same one used in the successful FP32 ALoDLM runs. Do not upgrade NumPy, torch or pandas within an initialized runtime. If installation warns about ABI incompatibility, restart the runtime, then run from this first code cell.

In [ ]:
import os, sys, subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
REPO_DIR='/content/ALoDLM_RSX28'
if not os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
print('Pinned official ALoDLM source:',CODE_COMMIT)

## 1. GPU, checkpoints, package imports and hard protocol identity

In [ ]:
import os, sys, gc, json, csv, re, math, ast, time, random, hashlib, zipfile, tempfile, statistics, shutil, platform, traceback, subprocess
from datetime import datetime,timezone
from pathlib import Path
from collections import defaultdict,Counter
from dataclasses import dataclass
import torch, numpy as np, pandas as pd, transformers
from scipy.stats import binomtest, beta
from huggingface_hub import snapshot_download
from types import SimpleNamespace
from google.colab import drive, files
import matplotlib.pyplot as plt
from IPython.display import display
if not torch.cuda.is_available(): raise RuntimeError('REQUIRES a real GPU: Colab > Runtime > Change runtime type > T4 GPU.')
if torch.cuda.get_device_properties(0).total_memory < 12*2**30: raise RuntimeError('Needs at least 12 GiB GPU VRAM.')
if torch.cuda.memory_allocated() > 2*2**30: raise RuntimeError('Restart runtime: unexpected other GPU allocations.')
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/RepairScopeX/NB28_Long_Adaptive_FP32_V1')
ROOT.mkdir(parents=True,exist_ok=True)
ART=ROOT/'artifacts'; ART.mkdir(exist_ok=True)
CACHE=ROOT/'records'; CACHE.mkdir(exist_ok=True)
MAX_SESSION_MINUTES=170
SESSION_START=time.monotonic()
MODEL_ID='amazon/ALoDLM-1.7B'
MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
PRECISION='fp32'; Q=0.4; TAU=0.4; WINDOW=16; MAX_NEW=96
GEN_MODE='left1'; REPAIR_MODE='entropy'
BASE_SEED=982611
CAL_TARGET={'countdown':8,'arithmetic':8,'multistep':8}
TEST_TARGET={'countdown':12,'arithmetic':12,'multistep':12}
ACTIONS=('global','tail25','tail50','tail75')
CAL_SEEDS=(13001,13002)
TEST_SEEDS=(14001,14002,14003)
STAGED_SEEDS=(15001,15002)
ABLATION_SEEDS=(16001,16002)
POLICIES=('global','best_fixed','adaptive','random_scope','tail50')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print('Torch',torch.__version__,'Transformers',transformers.__version__)

def canonical(v): return json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'),default=str)
def sha(v): return hashlib.sha256((v if isinstance(v,bytes) else canonical(v).encode())).hexdigest()
def atomic_write(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with tmp.open('w',encoding='utf8') as h:
        h.write(json.dumps(obj,ensure_ascii=False,sort_keys=True,indent=2,default=str)+'\n');h.flush();os.fsync(h.fileno())
    os.replace(tmp,path)
def rec_path(stage,tid):return CACHE/stage/(sha(tid)+'.json')
def save_rec(stage,rec):
    path=rec_path(stage,rec['trial_id'])
    if path.exists():
        old=json.loads(path.read_text());
        if sha(old)!=sha(rec):raise RuntimeError('Refusing to overwrite a different completed record: '+rec['trial_id'])
        return
    atomic_write(path,rec)
def get_rec(stage,tid):
    p=rec_path(stage,tid)
    if not p.exists():return None
    r=json.loads(p.read_text())
    if r['trial_id']!=tid or r.get('protocol_hash')!=RUN_SIG:raise RuntimeError('Checkpoint signature or identity mismatch '+tid)
    return r
def all_recs(stage):
    out=[]
    for p in sorted((CACHE/stage).glob('*.json')):
        r=json.loads(p.read_text())
        assert r['protocol_hash']==RUN_SIG
        out.append(r)
    assert len({r['trial_id'] for r in out})==len(out)
    return out
def stop_if_near_budget():
    used=(time.monotonic()-SESSION_START)/60
    if used >= MAX_SESSION_MINUTES:raise RuntimeError(f'Session wall-time safety stop ({used:.1f}/{MAX_SESSION_MINUTES} min). Export partial ZIP and resume in a new Colab session; earlier records are saved.')

PROTOCOL={'version':'RSX28-AloDLM-adaptive-v1','model':MODEL_ID,'model_rev':MODEL_REV,'code_commit':CODE_COMMIT,
          'fp':PRECISION,'q':Q,'tau':TAU,'window':WINDOW,'max_new':MAX_NEW,
          'gen_mode':GEN_MODE,'repair_mode':REPAIR_MODE,'cal_target':CAL_TARGET,'test_target':TEST_TARGET,
          'actions':ACTIONS,'cal_seeds':CAL_SEEDS,'test_seeds':TEST_SEEDS,
          'staged_seeds':STAGED_SEEDS,'ablation_seeds':ABLATION_SEEDS,'base_seed':BASE_SEED}
RUN_SIG=sha(PROTOCOL)
manifest=ROOT/'protocol.json'
if manifest.exists():
    existing=json.loads(manifest.read_text())
    if existing['hash']!=RUN_SIG or sha(existing['protocol'])!=sha(PROTOCOL):raise RuntimeError('Different experiment in this Drive path. NEVER overwrite or mix protocols. Use a new folder.')
else:atomic_write(manifest,{'hash':RUN_SIG,'protocol':PROTOCOL,'created_utc':datetime.now(timezone.utc).isoformat()})

## 2. Synthetic benchmark construction — prespecified, collision-checked and disjoint

In [ ]:
def make_problem(kind, rng, i, split):
    # Derived from deterministic synthetic templates, not GSM8K or SVAMP. No LLM-generated labels.
    if kind=='countdown':
        a,b,c=[rng.randint(3,30) for _ in range(3)]
        op=rng.choice(('plusminus','sum','prodminus','twice'))
        if op=='plusminus':target=a+b-c
        elif op=='sum':target=a+b+c
        elif op=='prodminus':target=a*b-c
        else:target=a*(b-c)
        prompt=f'Use the numbers {a}, {b}, and {c} exactly once to produce the target {target}. Return a valid arithmetic expression and end with `FINAL_ANSWER: <number>`.'
        return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[a,b,c],'template':op}
    if kind=='arithmetic':
        mode=i%4
        a=rng.randint(18,100);b=rng.randint(4,30)
        if mode==0:
            target=a+b;prompt=f'A collection contains {a} coins. A friend adds {b} more coins. How many coins are there now? End with FINAL_ANSWER: <number>.'
        elif mode==1:
            target=a-b;prompt=f'A shelf holds {a} books. Exactly {b} books are borrowed. How many books remain? End with FINAL_ANSWER: <number>.'
        elif mode==2:
            target=a*b;prompt=f'There are {a} identical boxes, each containing {b} pens. How many pens are there altogether? End with FINAL_ANSWER: <number>.'
        else:
            target=a;prompt=f'A number of stickers is divided equally among {b} children, and every child receives {a} stickers. How many stickers were there originally? End with FINAL_ANSWER: <number>.';target=a*b
        return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[],'template':mode}
    mode=i%4
    if mode==0:
        a=rng.randint(20,90);b=rng.randint(3,10);c=rng.randint(5,24);d=rng.randint(1,18)
        target=a+b*c-d;prompt=f'A warehouse begins with {a} packages. It receives {b} cartons with {c} packages each, then ships {d} packages. How many packages remain? End with FINAL_ANSWER: <number>.'
    elif mode==1:
        a=rng.randint(5,18);b=rng.randint(4,17);c=rng.randint(2,10);d=rng.randint(1,10)
        target=a*b+c*d;prompt=f'A school buys {a} packets containing {b} pencils each and {c} packets containing {d} pencils each. How many pencils in total? End with FINAL_ANSWER: <number>.'
    elif mode==2:
        a=rng.randint(3,12);b=rng.randint(5,20);c=rng.randint(3,25)
        target=a*b-c;prompt=f'A gardener plants {a} rows with {b} flowers each. Later {c} flowers are removed. How many flowers remain? End with FINAL_ANSWER: <number>.'
    else:
        a=rng.randint(10,40);b=rng.randint(2,8);c=rng.randint(5,20)
        target=a*b+c;prompt=f'A shop has {a} cartons of {b} bottles. A delivery adds {c} single bottles. How many bottles are available? End with FINAL_ANSWER: <number>.'
    return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[],'template':mode}

def build_bank(split,n_per_kind):
    out=[];seen=set()
    for k_idx,kind in enumerate(('countdown','arithmetic','multistep')):
        for i in range(n_per_kind):
            rng=random.Random(BASE_SEED + (0 if split=='calibration' else 1000000)+k_idx*10000+i*11)
            d=make_problem(kind,rng,i,split)
            fp=sha({'prompt':d['prompt'],'kind':kind})
            if fp in seen:raise ValueError('Duplicate within benchmark bank '+fp)
            seen.add(fp);d.update(split=split,problem_id=f'rsx28:{split}:{kind}:{i:04d}',problem_fingerprint=fp)
            out.append(d)
    return out
CAL_BANK=build_bank('calibration',40)
TEST_BANK=build_bank('heldout',56)
assert len(set(r['problem_fingerprint'] for r in CAL_BANK+TEST_BANK))==len(CAL_BANK+TEST_BANK)
BANK_HASH=sha(CAL_BANK+TEST_BANK)
BANK_PATH=ROOT/'frozen_problem_bank.json'
if BANK_PATH.exists():assert json.loads(BANK_PATH.read_text())['hash']==BANK_HASH
else:atomic_write(BANK_PATH,{'hash':BANK_HASH,'problems':CAL_BANK+TEST_BANK})
print('Frozen',len(CAL_BANK),'calibration candidates and',len(TEST_BANK),'heldout candidates; checksum',BANK_HASH)

## 3. Task grader and test-time eligibility signal — no word-problem answer leakage

In [ ]:
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
RUN_RE=re.compile(r'[0-9()+*×÷/\-+=\s]{3,}')
def closenum(a,b):return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-8)
def safe_expression(s):
    try:t=ast.parse(s,mode='eval')
    except Exception:return None
    lit=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):lit.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            a=walk(n.operand);return (-a if isinstance(n.op,ast.USub) else a)
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,ast.Add):return a+b
            if isinstance(n.op,ast.Sub):return a-b
            if isinstance(n.op,ast.Mult):return a*b
            if abs(b)<1e-12:raise ZeroDivisionError
            return a/b
        raise ValueError('Unsupported expression')
    try:
        v=walk(t)
        if math.isfinite(v):return {'value':v,'literals':lit}
    except Exception:pass
    return None

def extract_expression(text,problem):
    cleaned=text.replace('×','*').replace('÷','/')
    for m in RUN_RE.finditer(cleaned):
        raw=m.group(0)
        # Restrict to expression strings and avoid accidentally matching a target alone.
        for part in raw.split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            result=safe_expression(ex)
            if not result:continue
            if len(result['literals'])==len(problem['numbers']) and sorted(result['literals'])==sorted(map(float,problem['numbers'])) and closenum(result['value'],problem['target']):return ex
    return None

def final_number(text):
    found=FINAL_RE.findall(text)
    if found:return float(found[-1]),'explicit_final'
    found=ANSWER_RE.findall(text)
    if found:return float(found[-1]),'answer_phrase'
    # Deliberately do NOT use the last arbitrary numeral in unfinished reasoning.
    return None,'missing'

def grade_output(text,p):
    text=str(text);num,origin=final_number(text)
    final_lines=FINAL_RE.findall(text)
    strict=bool(final_lines) and closenum(float(final_lines[-1]),p['target'])
    if p['kind']=='countdown':
        expr=extract_expression(text,p);ok=expr is not None
        return {'success':ok,'numeric_correct':closenum(num,p['target']),'strict_correct':strict,'valid_expression':expr,'answer_origin':origin}
    return {'success':closenum(num,p['target']),'numeric_correct':closenum(num,p['target']),
            'strict_correct':strict,'valid_expression':None,'answer_origin':origin}

def public_accept(text,p):
    # May use Countdown target because task explicitly provides it; word problem gold answers are forbidden here.
    if p['kind']=='countdown':return extract_expression(str(text),p) is not None
    return bool(FINAL_RE.search(str(text)))

def features(text,p,n_tokens):
    # Target / correct answer NOT used as a predictive feature (including for Countdown).
    t=str(text);L=int(n_tokens)
    return {'kind':p['kind'],'length_bucket':'short' if L<45 else ('medium' if L<80 else 'long'),
            'has_final':bool(FINAL_RE.search(t)),'repetition':bool(re.search(r'(\b\w+\b)(?:\s+\1){2,}',t,re.I))}

def feat_bucket(f):return f["kind"]+'|'+('final' if f['has_final'] else 'no_final')

# Pure evaluator regression tests, including wrong-number and illegal-reuse rejections.
_p={'kind':'countdown','numbers':[2,3,4],'target':14}
assert grade_output('2 + 3 * 4 = 14',_p)['success']
assert not grade_output('2 + 4 + 4 = 14',_p)['success']
assert not grade_output('14',_p)['success']
assert not grade_output('2 + 3 * 4 = 14',{'kind':'countdown','numbers':[2,3,5],'target':14})['success']
_q={'kind':'arithmetic','target':14}
assert grade_output('FINAL_ANSWER: 14',_q)['success']
assert not grade_output('Then 14 times',_q)['success']
assert not grade_output('FINAL_ANSWER: 13',_q)['success']
print('Task verifiers: regression tests passed')

## 4. Model load + measured CUDA smoke test (no result is simulated)

In [ ]:
sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
assert MODEL_REV in str(MODEL_DIR) or Path(MODEL_DIR).is_dir()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
decoder=load_decoder(args)
model=decoder.model;tokenizer=decoder.tokenizer
assert next(model.parameters()).dtype==torch.float32
print('Model loaded FP32; CUDA allocated GiB:',round(torch.cuda.memory_allocated()/2**30,2))
PROMPT_SUFFIX='\n\nSolve carefully. For Countdown, show an arithmetic expression using all given numbers exactly once. End your response with FINAL_ANSWER: <number>.'
def encode_prompt(p):
    return list(tokenizer.apply_chat_template([{'role':'user','content':p['prompt']+PROMPT_SUFFIX}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
def decode_ids(ids):return tokenizer.decode(ids,skip_special_tokens=True).strip()
def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
@torch.inference_mode()
def invoke(prompt_ids,new_tokens,mode,seed):
    assert new_tokens>0 and new_tokens<=MAX_NEW
    stop_if_near_budget();set_seed(int(seed))
    cfg=DecodeConfig(q=Q,tau=TAU,mode=mode,window_size=WINDOW,max_new_tokens=int(new_tokens),position_penalty=None)
    cfg.validate()
    torch.cuda.reset_peak_memory_stats()
    started=time.perf_counter();result=decoder.generate(list(prompt_ids),cfg)
    ids=list(map(int,result['token_ids']))
    return {'token_ids':ids,'generated_tokens':int(result.get('generated_tokens',len(ids))),
            'outer_steps':int(result['outer_steps']),'recurrent_passes':int(result['recurrent_passes']),
            'loop_token_passes':int(result['loop_token_passes']),
            'mean_exit_depth':float(result['mean_exit_depth']) if result.get('mean_exit_depth') is not None else None,
            'wall_seconds':float(time.perf_counter()-started),'internal_wall_seconds':float(result.get('wall_seconds',0)),
            'stop_reason':str(result.get('stop_reason','unknown')),
            'peak_vram_gb':float(torch.cuda.max_memory_allocated()/2**30)}
smoke=invoke(encode_prompt(CAL_BANK[0]),12,GEN_MODE,BASE_SEED)
assert smoke['outer_steps']>0 and smoke['loop_token_passes']>0
print('REAL CUDA smoke PASSED', {k:smoke[k] for k in ('outer_steps','loop_token_passes','wall_seconds','peak_vram_gb')})

## 5. Generate model-native failures on frozen, disjoint problem banks
Exactly one seed per candidate, no repeated selection until the output fails; accumulate enough failures per type. Candidate identities are frozen **before** viewing outcomes. Do not adapt problem prompts to failures. Generations are saved individually and reused on resume.

In [ ]:
def natural_id(p):return p['problem_id']+'|natural|'+str(BASE_SEED)
def natural_run(p):
    tid=natural_id(p);old=get_rec('natural',tid)
    if old:return old
    r=invoke(encode_prompt(p),MAX_NEW,GEN_MODE,BASE_SEED)
    txt=decode_ids(r['token_ids']);v=grade_output(txt,p)
    obj={'trial_id':tid,'protocol_hash':RUN_SIG,'problem_id':p['problem_id'],'split':p['split'],
         'kind':p['kind'],'problem_fingerprint':p['problem_fingerprint'],'original_output':txt,
         'original_token_ids':r.pop('token_ids'),'evaluation':v,**r}
    save_rec('natural',obj)
    return obj

def cohort_from_natural(split):
    bank=CAL_BANK if split=='calibration' else TEST_BANK
    targets=CAL_TARGET if split=='calibration' else TEST_TARGET
    bykind=defaultdict(list)
    for p in bank:
        rec=get_rec('natural',natural_id(p))
        if rec and not rec['evaluation']['success']:bykind[p['kind']].append(p['problem_id'])
    if not all(len(bykind[k])>=targets[k] for k in targets):
        print('Eligible failures:',{k:len(bykind[k]) for k in targets},'required:',targets)
        return None
    subset=[x for k in targets for x in bykind[k][:targets[k]]]
    return subset

def natural_batch(split,start,limit=8):
    bank=CAL_BANK if split=='calibration' else TEST_BANK
    if cohort_from_natural(split):print('Already enough failures; skipping further candidate generation.');return
    for p in bank[start:start+limit]:
        if cohort_from_natural(split):break
        r=natural_run(p)
        print(p['problem_id'],r['evaluation']['success'],round(r['wall_seconds'],2),'s')
    print('Progress',len(all_recs('natural')),'native generations total.')

def freeze_cohorts():
    cp=cohort_from_natural('calibration');hp=cohort_from_natural('heldout')
    assert cp is not None and hp is not None,'Not enough model-native failures yet: continue natural-generation batches.'
    assert set(cp).isdisjoint(hp)
    content={'protocol_hash':RUN_SIG,'calibration':cp,'heldout':hp,'problem_bank_sha256':BANK_HASH}
    file=ROOT/'frozen_cohorts.json'
    if file.exists():assert json.loads(file.read_text())==content,'Frozen cohort differs; refusing test-set mutation.'
    else:atomic_write(file,content)
    print('FROZEN',len(cp),'calibration and',len(hp),'heldout native failures.')
    return content

### A. Calibration native-failure discovery (run cells in order; once quota met remaining cells are skipped)

In [ ]:
natural_batch('calibration',0,8)

In [ ]:
natural_batch('calibration',8,8)

In [ ]:
natural_batch('calibration',16,8)

In [ ]:
natural_batch('calibration',24,8)

In [ ]:
natural_batch('calibration',32,8)

In [ ]:
natural_batch('calibration',40,8)

In [ ]:
natural_batch('calibration',48,8)

In [ ]:
natural_batch('calibration',56,8)

In [ ]:
natural_batch('calibration',64,8)

In [ ]:
natural_batch('calibration',72,8)

In [ ]:
natural_batch('calibration',80,8)

In [ ]:
natural_batch('calibration',88,8)

In [ ]:
natural_batch('calibration',96,8)

In [ ]:
natural_batch('calibration',104,8)

In [ ]:
natural_batch('calibration',112,8)

### B. Held-out native-failure discovery (run cells in order; once quota met remaining cells are skipped)

In [ ]:
natural_batch('heldout',0,8)

In [ ]:
natural_batch('heldout',8,8)

In [ ]:
natural_batch('heldout',16,8)

In [ ]:
natural_batch('heldout',24,8)

In [ ]:
natural_batch('heldout',32,8)

In [ ]:
natural_batch('heldout',40,8)

In [ ]:
natural_batch('heldout',48,8)

In [ ]:
natural_batch('heldout',56,8)

In [ ]:
natural_batch('heldout',64,8)

In [ ]:
natural_batch('heldout',72,8)

In [ ]:
natural_batch('heldout',80,8)

In [ ]:
natural_batch('heldout',88,8)

In [ ]:
natural_batch('heldout',96,8)

In [ ]:
natural_batch('heldout',104,8)

In [ ]:
natural_batch('heldout',112,8)

In [ ]:
natural_batch('heldout',120,8)

In [ ]:
natural_batch('heldout',128,8)

In [ ]:
natural_batch('heldout',136,8)

In [ ]:
natural_batch('heldout',144,8)

In [ ]:
natural_batch('heldout',152,8)

In [ ]:
natural_batch('heldout',160,8)

### C. Freeze cohort; run only after the two quotas are met

In [ ]:
COHORTS=freeze_cohorts()
PROBLEMS={p['problem_id']:p for p in CAL_BANK+TEST_BANK}
def failure(pid):
    p=PROBLEMS[pid];r=get_rec('natural',natural_id(p))
    assert r and not r['evaluation']['success'] and r['problem_fingerprint']==p['problem_fingerprint']
    return p,r

## 6. Native prefix-conditioned suffix repair; preservation and compute audits

In [ ]:
def scope_cut(n,action):
    if action=='global':return 0
    if action not in ACTIONS:raise KeyError(action)
    frac={'tail25':.75,'tail50':.50,'tail75':.25}[action]
    return max(1,min(n-1,int(math.floor(n*frac))))

def repair_one(pid,action,seed,mode=REPAIR_MODE,stage='calibration'):
    assert action in ACTIONS
    tid=f'{stage}|{pid}|{action}|{mode}|{seed}'
    old=get_rec(stage,tid)
    if old:return old
    p,source=failure(pid)
    ids=source['original_token_ids']; n=len(ids);assert n>=4
    cut=scope_cut(n,action);preserved=ids[:cut];budget=max(1,n-cut)
    assert budget<=MAX_NEW
    r=invoke(encode_prompt(p)+preserved,budget,mode,seed)
    new_full=preserved+r['token_ids']
    assert new_full[:cut]==preserved, 'Prefix-preservation invariant failed.'
    txt=decode_ids(new_full)
    eval_=grade_output(txt,p)
    obj={'trial_id':tid,'protocol_hash':RUN_SIG,'stage':stage,'problem_id':pid,'split':p['split'],
        'kind':p['kind'],'action':action,'mode':mode,'seed':int(seed),'original_response_tokens':n,
        'preserved_tokens':cut,'requested_new_tokens':budget,'preserved_fraction':cut/n,
        'exact_prefix_preserved':True,'original_output':source['original_output'],'repaired_output':txt,
        'evaluation':eval_,'public_accept':public_accept(txt,p),
        'original_repeated_exactly':txt==source['original_output'],
        **{k:v for k,v in r.items() if k!='token_ids'}}
    save_rec(stage,obj)
    return obj

def run_grid_batch(stage,grid,start,count=8):
    for pid,action,mode,seed in grid[start:start+count]:
        r=repair_one(pid,action,seed,mode,stage)
        print(f'{stage} {r["problem_id"]} {action} {mode} {seed}: success={r["evaluation"]["success"]} LTP={r["loop_token_passes"]}')
    print('Saved',len(all_recs(stage)),'stage records')

CAL_GRID=[(pid,a,REPAIR_MODE,s) for pid in COHORTS['calibration'] for a in ACTIONS for s in CAL_SEEDS]
assert len(CAL_GRID)==192

## 7. Calibration: 24 native failures × 4 repair actions × 2 seeds = 192 real repairs

In [ ]:
run_grid_batch('calibration',CAL_GRID,0,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,8,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,16,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,24,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,32,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,40,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,48,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,56,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,64,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,72,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,80,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,88,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,96,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,104,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,112,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,120,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,128,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,136,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,144,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,152,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,160,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,168,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,176,8)

In [ ]:
run_grid_batch('calibration',CAL_GRID,184,8)

## 8. Fit and freeze adaptive policy from calibration ONLY

In [ ]:
def calibrate():
    data=all_recs('calibration');exp={f'calibration|{pid}|{a}|{REPAIR_MODE}|{seed}' for pid in COHORTS['calibration'] for a in ACTIONS for seed in CAL_SEEDS}
    assert len(data)==192 and set(r['trial_id'] for r in data)==exp,'Calibration incomplete; DO NOT inspect heldout results or fit policy yet.'
    pools=defaultdict(list)
    for r in data:
        p,src=failure(r['problem_id'])
        f=features(src['original_output'],p,len(src['original_token_ids']))
        pools[(feat_bucket(f),r['action'])].append(r)
        pools[(p['kind'],r['action'])].append(r)
        pools[('all',r['action'])].append(r)
    # Predeclared Laplace prior and lightweight cost penalty. No target-label features at deployment.
    def value(group,action):
        rows=pools.get((group,action),[])
        if not rows:return None
        succ=sum(r['evaluation']['success'] for r in rows)
        ltp=statistics.mean(r['loop_token_passes'] for r in rows)
        glob=pools[('all','global')]
        relative_cost=ltp/max(1,statistics.mean(r['loop_token_passes'] for r in glob))
        return (succ+1)/(len(rows)+2)-0.05*relative_cost
    fallback=max(ACTIONS,key=lambda a:(value('all',a),-ACTIONS.index(a)))
    kinds=sorted({p['kind'] for p in CAL_BANK})
    mapping={}
    for kind in kinds:
        for has_final in (True,False):
            buck=kind+'|'+('final' if has_final else 'no_final')
            group=buck if min(len(pools[(buck,a)]) for a in ACTIONS)>=6 else kind
            mapping[buck]=max(ACTIONS,key=lambda a:(value(group,a),-ACTIONS.index(a)))
    calib_summary={a:{'trials':len(pools[('all',a)]),'successes':sum(r['evaluation']['success'] for r in pools[('all',a)]),
                      'mean_loop_token_passes':statistics.mean(r['loop_token_passes'] for r in pools[('all',a)])} for a in ACTIONS}
    frozen={'protocol_hash':RUN_SIG,'calibration_trial_hash':sha(sorted((r['trial_id'],r['evaluation']['success']) for r in data)),
            'best_fixed':fallback,'mapping':mapping,'lambda_cost':0.05,'action_summary':calib_summary,'trained_only_on':'calibration'}
    path=ROOT/'FROZEN_POLICY.json'
    if path.exists():assert json.loads(path.read_text())==frozen,'Cannot refit a different policy after freezing.'
    else:atomic_write(path,frozen)
    return frozen
POLICY=calibrate()
print('FROZEN POLICY:',json.dumps(POLICY,indent=2))

## 9. Held-out single-proposal comparison — **five** policies × 36 independent failures × 3 seeds = 540 repairs

In [ ]:
def choose_action(policy,pid,seed):
    p,src=failure(pid)
    if policy=='global':return 'global'
    if policy=='tail50':return 'tail50'
    if policy=='best_fixed':return POLICY['best_fixed']
    if policy=='adaptive':
        f=features(src['original_output'],p,len(src['original_token_ids']))
        return POLICY['mapping'][feat_bucket(f)]
    if policy=='random_scope':
        # Precommitted randomization independent of targets/outcomes; includes global.
        seed_int=int(sha({'pid':pid,'seed':seed,'policy':'random_scope'})[:12],16)
        return random.Random(seed_int).choice(ACTIONS)
    raise KeyError(policy)
def heldout_one(pid,policy,seed):
    action=choose_action(policy,pid,seed)
    primitive=repair_one(pid,action,seed,REPAIR_MODE,stage='test_primitives')
    tid=f'heldout|{pid}|{policy}|{seed}'
    old=get_rec('heldout',tid)
    if old:return old
    r={'trial_id':tid,'protocol_hash':RUN_SIG,'problem_id':pid,'policy':policy,'action':action,'seed':seed,
       'primitive_id':primitive['trial_id'],'evaluation':primitive['evaluation'],'loop_token_passes':primitive['loop_token_passes'],
       'recurrent_passes':primitive['recurrent_passes'],'wall_seconds':primitive['wall_seconds'],
       'preserved_fraction':primitive['preserved_fraction'],'public_accept':primitive['public_accept'],
       'original_repeated_exactly':primitive['original_repeated_exactly']}
    save_rec('heldout',r)
    return r
TEST_GRID=[(pid,pol,s) for pid in COHORTS['heldout'] for pol in POLICIES for s in TEST_SEEDS]
assert len(TEST_GRID)==540
# Policies may share the same physical candidate (e.g. adaptive=global). This is logged as
# a shared primitive, never represented as two independent GPU samples or charged twice in resources.
def heldout_batch(start,count=8):
    for pid,pol,s in TEST_GRID[start:start+count]:
        r=heldout_one(pid,pol,s)
        print(pid,pol,r['action'],s,'SUCCESS',r['evaluation']['success'],'LTP',r['loop_token_passes'])
    print('Heldout logical outcomes',len(all_recs('heldout')),'/',len(TEST_GRID),
          '| unique physical GPU candidates',len(all_recs('test_primitives')))

In [ ]:
heldout_batch(0,8)

In [ ]:
heldout_batch(8,8)

In [ ]:
heldout_batch(16,8)

In [ ]:
heldout_batch(24,8)

In [ ]:
heldout_batch(32,8)

In [ ]:
heldout_batch(40,8)

In [ ]:
heldout_batch(48,8)

In [ ]:
heldout_batch(56,8)

In [ ]:
heldout_batch(64,8)

In [ ]:
heldout_batch(72,8)

In [ ]:
heldout_batch(80,8)

In [ ]:
heldout_batch(88,8)

In [ ]:
heldout_batch(96,8)

In [ ]:
heldout_batch(104,8)

In [ ]:
heldout_batch(112,8)

In [ ]:
heldout_batch(120,8)

In [ ]:
heldout_batch(128,8)

In [ ]:
heldout_batch(136,8)

In [ ]:
heldout_batch(144,8)

In [ ]:
heldout_batch(152,8)

In [ ]:
heldout_batch(160,8)

In [ ]:
heldout_batch(168,8)

In [ ]:
heldout_batch(176,8)

In [ ]:
heldout_batch(184,8)

In [ ]:
heldout_batch(192,8)

In [ ]:
heldout_batch(200,8)

In [ ]:
heldout_batch(208,8)

In [ ]:
heldout_batch(216,8)

In [ ]:
heldout_batch(224,8)

In [ ]:
heldout_batch(232,8)

In [ ]:
heldout_batch(240,8)

In [ ]:
heldout_batch(248,8)

In [ ]:
heldout_batch(256,8)

In [ ]:
heldout_batch(264,8)

In [ ]:
heldout_batch(272,8)

In [ ]:
heldout_batch(280,8)

In [ ]:
heldout_batch(288,8)

In [ ]:
heldout_batch(296,8)

In [ ]:
heldout_batch(304,8)

In [ ]:
heldout_batch(312,8)

In [ ]:
heldout_batch(320,8)

In [ ]:
heldout_batch(328,8)

In [ ]:
heldout_batch(336,8)

In [ ]:
heldout_batch(344,8)

In [ ]:
heldout_batch(352,8)

In [ ]:
heldout_batch(360,8)

In [ ]:
heldout_batch(368,8)

In [ ]:
heldout_batch(376,8)

In [ ]:
heldout_batch(384,8)

In [ ]:
heldout_batch(392,8)

In [ ]:
heldout_batch(400,8)

In [ ]:
heldout_batch(408,8)

In [ ]:
heldout_batch(416,8)

In [ ]:
heldout_batch(424,8)

In [ ]:
heldout_batch(432,8)

In [ ]:
heldout_batch(440,8)

In [ ]:
heldout_batch(448,8)

In [ ]:
heldout_batch(456,8)

In [ ]:
heldout_batch(464,8)

In [ ]:
heldout_batch(472,8)

In [ ]:
heldout_batch(480,8)

In [ ]:
heldout_batch(488,8)

In [ ]:
heldout_batch(496,8)

In [ ]:
heldout_batch(504,8)

In [ ]:
heldout_batch(512,8)

In [ ]:
heldout_batch(520,8)

In [ ]:
heldout_batch(528,8)

In [ ]:
heldout_batch(536,8)

## 10. Two-attempt strategy comparison: adaptive→global vs global→global; no held-out label selection

In [ ]:
STAGE_ARMS=('adaptive_then_global','global_then_global')
STAGE_GRID=[(pid,arm,s) for pid in COHORTS['heldout'] for arm in STAGE_ARMS for s in STAGED_SEEDS]
assert len(STAGE_GRID)==144

def staged_one(pid,arm,seed):
    tid=f'staged|{pid}|{arm}|{seed}'
    old=get_rec('staged',tid)
    if old:return old
    p,src=failure(pid)
    actions=(choose_action('adaptive',pid,seed),'global') if arm=='adaptive_then_global' else ('global','global')
    calls=[];accepted=None
    for j,action in enumerate(actions):
        # Distinct seeds across two attempts; no test-time gold answers involved in stopping.
        attempt_seed=seed+j*777
        primitive=repair_one(pid,action,attempt_seed,REPAIR_MODE,stage='staged_primitives')
        calls.append(primitive)
        if primitive['public_accept']:
            accepted=primitive;break
    # If neither meets the public gate, the last candidate is the output (not oracle-selected).
    selected=accepted if accepted else calls[-1]
    r={'trial_id':tid,'protocol_hash':RUN_SIG,'problem_id':pid,'arm':arm,'seed':seed,
       'actions_used':[x['action'] for x in calls],'proposal_count':len(calls),
       'proposal_ids':[x['trial_id'] for x in calls],
       'public_accept':selected['public_accept'],'evaluation':selected['evaluation'],
       'loop_token_passes':sum(x['loop_token_passes'] for x in calls),
       'wall_seconds':sum(x['wall_seconds'] for x in calls),
       'recurrent_passes':sum(x['recurrent_passes'] for x in calls)}
    save_rec('staged',r)
    return r

def staged_batch(start,count=6):
    for pid,arm,s in STAGE_GRID[start:start+count]:
        r=staged_one(pid,arm,s)
        print(pid,arm,s,'success',r['evaluation']['success'],'proposals',r['proposal_count'],
              'loop_token_passes',r['loop_token_passes'])
    print('Staged',len(all_recs('staged')),'/',len(STAGE_GRID))

In [ ]:
staged_batch(0,6)

In [ ]:
staged_batch(6,6)

In [ ]:
staged_batch(12,6)

In [ ]:
staged_batch(18,6)

In [ ]:
staged_batch(24,6)

In [ ]:
staged_batch(30,6)

In [ ]:
staged_batch(36,6)

In [ ]:
staged_batch(42,6)

In [ ]:
staged_batch(48,6)

In [ ]:
staged_batch(54,6)

In [ ]:
staged_batch(60,6)

In [ ]:
staged_batch(66,6)

In [ ]:
staged_batch(72,6)

In [ ]:
staged_batch(78,6)

In [ ]:
staged_batch(84,6)

In [ ]:
staged_batch(90,6)

In [ ]:
staged_batch(96,6)

In [ ]:
staged_batch(102,6)

In [ ]:
staged_batch(108,6)

In [ ]:
staged_batch(114,6)

In [ ]:
staged_batch(120,6)

In [ ]:
staged_batch(126,6)

In [ ]:
staged_batch(132,6)

In [ ]:
staged_batch(138,6)

## 11. Decoder-policy ablation: held-out left1 × (global, tail50) × two seeds (144 repairs)

In [ ]:
ABLATION_GRID=[(pid,a,s) for pid in COHORTS['heldout'] for a in ('global','tail50') for s in ABLATION_SEEDS]
assert len(ABLATION_GRID)==144
def ablate_batch(start,count=8):
    for pid,a,s in ABLATION_GRID[start:start+count]:
        r=repair_one(pid,a,s,'left1',stage='left1_ablation')
        print(pid,a,s,'success',r['evaluation']['success'],'native_LTP',r['loop_token_passes'])
    print('Left1 ablation',len(all_recs('left1_ablation')),'/',len(ABLATION_GRID))

In [ ]:
ablate_batch(0,8)

In [ ]:
ablate_batch(8,8)

In [ ]:
ablate_batch(16,8)

In [ ]:
ablate_batch(24,8)

In [ ]:
ablate_batch(32,8)

In [ ]:
ablate_batch(40,8)

In [ ]:
ablate_batch(48,8)

In [ ]:
ablate_batch(56,8)

In [ ]:
ablate_batch(64,8)

In [ ]:
ablate_batch(72,8)

In [ ]:
ablate_batch(80,8)

In [ ]:
ablate_batch(88,8)

In [ ]:
ablate_batch(96,8)

In [ ]:
ablate_batch(104,8)

In [ ]:
ablate_batch(112,8)

In [ ]:
ablate_batch(120,8)

In [ ]:
ablate_batch(128,8)

In [ ]:
ablate_batch(136,8)

## 12. Mandatory integrity checks, paired problem-level inference, publication tables
No analysis runs on partial experiments. If Colab interrupts, use **Section 13** partial export, reconnect, and complete missing batches. Multiple seeds are clustered by distinct problem, not counted as independent sample size. No result is declared statistically significant without two-sided paired evidence.

In [ ]:
import itertools

def exact_ci(successes,n,alpha=.05):
    if n<=0:return (None,None)
    return ((0 if successes==0 else float(beta.ppf(alpha/2,successes,n-successes+1))),
            (1 if successes==n else float(beta.ppf(1-alpha/2,successes+1,n-successes))))
def paired_problem_stats(rows,name,arm1,arm2,seed_col='seed',arm_col='policy'):
    by=defaultdict(dict)
    for r in rows:
        if r[arm_col] not in (arm1,arm2):continue
        by[r['problem_id']][r[arm_col],r[seed_col]]=bool(r['evaluation']['success'])
    n=0;wins=losses=ties=0;paired=[]
    for pid,d in sorted(by.items()):
        sa=any(v for (a,s),v in d.items() if a==arm1)
        sb=any(v for (a,s),v in d.items() if a==arm2)
        if not {arm1,arm2}.issubset({a for a,s in d}):raise AssertionError('Paired missing arm')
        wins+=sa and not sb;losses+=sb and not sa;ties+=sa==sb;n+=1
        paired.append((pid,int(sa),int(sb)))
    nd=wins+losses;p=float(binomtest(wins,nd,.5).pvalue) if nd else 1.0
    # Cluster bootstrap on independent *problem IDs*, with zero-width observed CI
    # clearly labeled a conditional empirical bootstrap rather than coverage proof.
    rr=random.Random(BASE_SEED+188);dif=[]
    for i in range(3000):
        sample=[rr.choice(paired) for _ in paired]
        dif.append(sum(x[1]-x[2] for x in sample)/len(sample))
    dif.sort()
    return {'comparison':name,'n_problems':n,'A':arm1,'B':arm2,'A_only_problems':int(wins),
            'B_only_problems':int(losses),'both_or_neither':int(ties),'exact_twosided_p':p,
            'observed_problem_rate_difference':(wins-losses)/n,
            'conditional_cluster_bootstrap_low':dif[74],'conditional_cluster_bootstrap_high':dif[2924],
            'bootstrap_warning':'conditional empirical bootstrap; zero-width intervals with zero events do not imply certainty'}
def stage_check():
    required={'calibration':len(CAL_GRID),'heldout':len(TEST_GRID),'staged':len(STAGE_GRID),'left1_ablation':len(ABLATION_GRID)}
    actual={s:len(all_recs(s)) for s in required}
    missing={s:(required[s]-actual[s]) for s in required}
    print('REQUIRED',required,'ACTUAL',actual,'MISSING',missing)
    assert all(v==0 for v in missing.values()),'Incomplete! Do not freeze partial research tables.'
    assert (ROOT/'FROZEN_POLICY.json').exists()
    assert (ROOT/'frozen_cohorts.json').exists()
    for stage in required:
        for r in all_recs(stage):assert r['protocol_hash']==RUN_SIG
    for stage in ('calibration','test_primitives','staged_primitives','left1_ablation'):
        for r in all_recs(stage):assert r['exact_prefix_preserved']
    assert len(COHORTS['calibration'])==24 and len(COHORTS['heldout'])==36
    return actual
COUNTS=stage_check()
HELD=all_recs('heldout');STAGED=all_recs('staged');CAL=all_recs('calibration');AB=all_recs('left1_ablation')

def summarize(rows,groupfield):
    ans=[]
    for name in sorted(set(r[groupfield] for r in rows)):
        group=[r for r in rows if r[groupfield]==name]
        n=len(group);success=sum(r['evaluation']['success'] for r in group)
        unique=len({r['problem_id'] for r in group if r['evaluation']['success']})
        low,hi=exact_ci(success,n)
        ans.append({groupfield:name,'trials':n,'task_successes':success,'task_success_rate':success/n,
            'independent_test_problems':len({r['problem_id'] for r in group}),
            'distinct_recovered_problems':unique,'trial_exact95_low':low,'trial_exact95_high':hi,
            'mean_native_loop_token_passes':statistics.mean(r['loop_token_passes'] for r in group),
            'median_native_loop_token_passes':statistics.median(r['loop_token_passes'] for r in group),
            'mean_wall_seconds':statistics.mean(r['wall_seconds'] for r in group)})
    return pd.DataFrame(ans)
CAL_SUM=summarize(CAL,'action');H_SUM=summarize(HELD,'policy');S_SUM=summarize(STAGED,'arm');A_SUM=summarize(AB,'action')
PTEST=[paired_problem_stats(HELD,'single: adaptive versus '+p,'adaptive',p) for p in ('global','best_fixed','random_scope','tail50')]
PTEST.append(paired_problem_stats(STAGED,'two attempts: adaptive→global versus global→global','adaptive_then_global','global_then_global',arm_col='arm'))
# Decoder contrasts at the same held-out IDs but different seeds are comparative, not strictly seed-paired.
PTEST_DF=pd.DataFrame(PTEST)
# Holm correction across the five prespecified primary paired comparisons.
pordered=sorted(range(len(PTEST)),key=lambda i:PTEST[i]['exact_twosided_p'])
corrected=[0]*len(PTEST);prev=0
for rank,i in enumerate(pordered):
    prev=max(prev,min(1.0,(len(PTEST)-rank)*PTEST[i]['exact_twosided_p']))
    corrected[i]=prev
PTEST_DF['holm_p']=corrected
for filename,df in [('calibration_by_scope.csv',CAL_SUM),('heldout_by_policy.csv',H_SUM),('staged_retry_comparison.csv',S_SUM),('left1_decoder_ablation.csv',A_SUM),('paired_problem_tests.csv',PTEST_DF)]:
    df.to_csv(ART/filename,index=False)
for stage in ('natural','calibration','heldout','test_primitives','staged','staged_primitives','left1_ablation'):
    rows=all_recs(stage)
    with (ART/(stage+'_all_records.jsonl')).open('w',encoding='utf8') as f:
        for r in rows:f.write(canonical(r)+'\n')

fig,ax=plt.subplots(figsize=(8,4.5))
vals=H_SUM.set_index('policy').loc[list(POLICIES)]
ax.bar(range(len(vals)),vals['distinct_recovered_problems'])
ax.set_xticks(range(len(vals)),vals.index,rotation=18,ha='right');ax.set_ylabel('Distinct held-out problems solved (36 total)')
ax.set_title('Single-proposal repair: independent problem recovery')
fig.tight_layout();fig.savefig(ART/'fig01_heldout_problem_recovery.pdf');fig.savefig(ART/'fig01_heldout_problem_recovery.png',dpi=180);plt.close(fig)
fig,ax=plt.subplots(figsize=(7,4.4))
for _,r in H_SUM.iterrows():ax.scatter(r['mean_native_loop_token_passes'],r['task_success_rate'],s=80,label=r['policy'])
ax.set_xlabel('Mean native decoder loop-token passes / proposal');ax.set_ylabel('Task-success rate (trial-level)')
ax.set_title('Compute–success frontier (NOT FLOP-matched)');ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(ART/'fig02_compute_success.pdf');fig.savefig(ART/'fig02_compute_success.png',dpi=180);plt.close(fig)
summary={'protocol':PROTOCOL,'protocol_hash':RUN_SIG,'frozen_policy':POLICY,'cohorts':COHORTS,
         'counts':COUNTS,'natural_trials':len(all_recs('natural')),
         'unique_physical_test_primitives':len(all_recs('test_primitives')),
         'staged_primitive_calls':len(all_recs('staged_primitives')),
         'notes':['Prospective disjoint synthetic data; no claim of GSM8K/SVAMP generalization.',
                  'Multiple policies can reuse one identical physical candidate; evaluate statistical dependence by problem.',
                  'Word-problem public stopping verifier checks final-answer format, NOT ground truth.',
                  'Candidate count held constant; native model compute is measured, NOT necessarily equal.',
                  'No result of prior Notebook 21 is pooled here.']}
atomic_write(ART/'research_manifest.json',summary)
print('EVIDENCE GATE PASS; heldout summary:');display(H_SUM)
print('Problem-level paired tests:');display(PTEST_DF)

### 12a. Additional publication diagnostics: calibrated compute budgets, failure recurrence and dataset stratification

These are **post hoc analyses of the frozen held-out outputs**, not extra candidate generation and not causal compute-matched interventions. Resource thresholds are fixed from calibration global-repair observations. An over-budget candidate is classified as infeasible without generating replacements. Evaluate true cost-matching prospectively in future work.

In [ ]:

# The paper must distinguish observed correct answers, inherited wrong answers, and false-positive stopping signals.
ORIGINALS={pid:failure(pid) for pid in COHORTS['heldout']}
primitives=all_recs('test_primitives')
persist=[]
for r in primitives:
    pid=r['problem_id'];p,source=ORIGINALS[pid]
    old_num,old_origin=final_number(source['original_output'])
    new_num,new_origin=final_number(r['repaired_output'])
    old_was_wrong=old_num is not None and not closenum(old_num,p['target'])
    persist.append({'problem_id':pid,'kind':p['kind'],'action':r['action'],'seed':r['seed'],
        'old_explicit_answer_origin':old_origin,'new_explicit_answer_origin':new_origin,
        'old_answer_was_wrong':old_was_wrong,'wrong_numeric_recurred':bool(old_was_wrong and
                new_num is not None and closenum(old_num,new_num) and not r['evaluation']['success']),
        'exact_text_recurred':r['original_repeated_exactly'],'task_success':r['evaluation']['success'],
        'preserved_fraction':r['preserved_fraction'],'loop_token_passes':r['loop_token_passes']})
PERSIST=pd.DataFrame(persist)
PERSIST.to_csv(ART/'wrong_answer_persistence_raw.csv',index=False)
rows=[]
for action,grp in PERSIST.groupby('action'):
    old_wrong=grp[grp['old_answer_was_wrong']]
    rows.append({'action':action,'physical_trials':len(grp),'original_explicit_wrong_count':len(old_wrong),
                 'recurrence_given_explicit_wrong':(float(old_wrong['wrong_numeric_recurred'].mean()) if len(old_wrong) else None),
                 'exact_text_recurrence_rate':float(grp['exact_text_recurred'].mean()),
                 'mean_preserved_fraction':float(grp['preserved_fraction'].mean())})
PERSIST_SUM=pd.DataFrame(rows)
PERSIST_SUM.to_csv(ART/'wrong_answer_persistence_by_action.csv',index=False)
strat=[]
for (policy,kind),grp in pd.DataFrame([{'policy':r['policy'],'kind':PROBLEMS[r['problem_id']]['kind'],
            'success':int(r['evaluation']['success']),'pid':r['problem_id']} for r in HELD]).groupby(['policy','kind']):
    strat.append({'policy':policy,'kind':kind,'trials':len(grp),'successful_trials':int(grp['success'].sum()),
                  'distinct_recovered_problems':len(set(grp.loc[grp['success']==1,'pid']))})
pd.DataFrame(strat).to_csv(ART/'heldout_by_policy_and_task.csv',index=False)
cal_global=sorted(r['loop_token_passes'] for r in CAL if r['action']=='global')
assert cal_global
quantiles={q:int(cal_global[max(0,min(len(cal_global)-1,round(q*(len(cal_global)-1))))]) for q in (.25,.50,.75,1.0)}
BUDGET_ROWS=[];BUDGET_TESTS=[]
for pct,budget in quantiles.items():
    altered=[]
    for r in HELD:
        c={**r,'evaluation':{**r['evaluation'],'success':bool(r['evaluation']['success'] and r['loop_token_passes']<=budget)}}
        altered.append(c)
    for pol in POLICIES:
        sub=[r for r in altered if r['policy']==pol]
        BUDGET_ROWS.append({'calibration_global_quantile':pct,'native_loop_token_passes_cap':budget,
                           'policy':pol,'feasible_proposals':sum(r['loop_token_passes']<=budget for r in HELD if r['policy']==pol),
                           'successful_eligible_trials':sum(r['evaluation']['success'] for r in sub),
                           'distinct_problems_with_eligible_success':len({r['problem_id'] for r in sub if r['evaluation']['success']})})
    BUDGET_TESTS.append({'calibration_global_quantile':pct,'native_loop_token_passes_cap':budget,
                  **paired_problem_stats(altered,'budget-constrained adaptive/global','adaptive','global')})
pd.DataFrame(BUDGET_ROWS).to_csv(ART/'posthoc_cost_feasibility_by_policy.csv',index=False)
pd.DataFrame(BUDGET_TESTS).to_csv(ART/'posthoc_cost_feasibility_paired_tests.csv',index=False)
print('Additional diagnostics saved: persistence, task stratification, posthoc compute budget feasibility.')


## 13. Partial and final ZIP export — RUN ANY TIME, including after a disconnect (CPU only)
Exports immutable checkpoint copies, frozen bank/cohorts, calibration policy, and research outputs present. Does **not** fabricate missing stages. On a partially completed run, `partial=true`; the final manuscript should use only a fully validated run. Files may be large due to original outputs.

In [ ]:
def export_bundle():
    out=Path('/content/RepairScopeX_NB28_Long_Adaptive_FP32_results.zip')
    stages=('natural','calibration','heldout','test_primitives','staged','staged_primitives','left1_ablation')
    status={s:len(all_recs(s)) for s in stages}
    required={'calibration':192,'heldout':540,'staged':144,'left1_ablation':144}
    finished=(all(status[k]==v for k,v in required.items()) and (ROOT/'FROZEN_POLICY.json').exists() and (ART/'research_manifest.json').exists() and (ART/'posthoc_cost_feasibility_by_policy.csv').exists())
    aux=[]
    for p in (ROOT/'protocol.json',ROOT/'frozen_problem_bank.json',ROOT/'frozen_cohorts.json',ROOT/'FROZEN_POLICY.json'):
        if p.exists():aux.append((p,p.name))
    for p in sorted(ART.glob('*')):aux.append((p,'artifacts/'+p.name))
    for stage in stages:
        for p in sorted((CACHE/stage).glob('*.json')):aux.append((p,'records/'+stage+'/'+p.name))
    checksum={name:hashlib.sha256(p.read_bytes()).hexdigest() for p,name in aux}
    exportmeta={'protocol_hash':RUN_SIG,'partial':not finished,'stage_counts':status,'required':required,
        'source_git_commit':CODE_COMMIT,'model_revision':MODEL_REV,'file_checksums_sha256':checksum}
    # Fail closed on duplicate archive member names and verify bytes after writing.
    assert len({n for p,n in aux})==len(aux)
    with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        z.writestr('EXPORT_STATUS.json',json.dumps(exportmeta,indent=2))
        for p,name in aux:z.write(p,name)
    with zipfile.ZipFile(out) as z:
        assert z.testzip() is None
        for name,expected in checksum.items():assert hashlib.sha256(z.read(name)).hexdigest()==expected
    print('EXPORTED',out,'partial=',not finished,'MB=',round(out.stat().st_size/2**20,2))
    return out
ZIP=export_bundle()
files.download(str(ZIP))

## 14. Interpreting this experiment correctly

- Success means **task correctness under the explicit grader**, not merely a plausible number somewhere in a response; the word-problem grader accepts explicit final-answer or clear "answer is" phrases. Manual spot checks are recommended before publication.
- **Adaptive policy** is learned only on the calibration split. The held-out split is generated with disjoint seeds and unique prompts; no re-selection after inspecting its results.
- **Candidate-count matched** is not identical to compute matched: denoising depends on suffix length, adaptive-depth exits, and native loop-token passes. Compare performance against cost and avoid unconditional superiority claims.
- A programmatic Countdown validator has access to its **target in the prompt**, whereas the word-problem stopping verifier deliberately does not see the held-out numerical target. These are different validation regimes.
- Full FP32 and source/checkpoint revisions are pinned. `left1` versus `entropy` can change *both* generation dynamics and repair behavior; the comparison is an ablation, not a causal proof.
- Some fixed and adaptive policies may choose the same action and generate the same physical candidate at the same seed. Those outputs are **shared observations**, logged as common physical primitive IDs, not counted as independent GPU experiments.
- The prompt collection is **synthetic procedural math**, with limited generalization claims. The experiment is exploratory unless a fresh external benchmark is subsequently used.
- This notebook records a maximum of 170 minutes in one runtime. If time runs out, download the partial ZIP; checkpoints remain in Drive. A 3-hour Colab Free GPU allocation is **not guaranteed**.
- Never choose a best action based on the held-out scores and then re-report it as prespecified. Keep exploratory oracle performance, if computed, separate.